## 0. Setup (run first in every notebook)
Attach these inputs via **Add Input**: your uploaded `causal-rag-kaggle` dataset (this project) and, depending on the stage,
the Lending Club dataset (`wordsforthewise/lending-club`) and/or the artifacts dataset saved from the previous notebook.
Turn **Internet ON** (policy documents, models, RCT data) and set the accelerator to **GPU T4 x2** for stages 04-08.

In [ ]:
!pip -q install causal-learn econml sentence-transformers bitsandbytes accelerate gradio pypdf beautifulsoup4 scikit-uplift pyarrow 2>&1 | tail -1
import sys, os, glob
hits = glob.glob('/kaggle/input/**/causal_rag/__init__.py', recursive=True)
print('causal_rag copies found:'); [print('  ', h) for h in hits]
need = ('data_prep.py', 'causal_graph.py', 'retrieval.py', 'counterfactual.py', 'audit.py', 'compare.py', 'pipeline.py', 'live_demo.py')
complete = [h for h in hits if all(os.path.exists(os.path.join(os.path.dirname(h), n)) for n in need)]
v11 = [h for h in complete if 'prompt_graded' in open(os.path.join(os.path.dirname(h), 'prompts.py')).read()]
if not complete:
    for h in hits: print('FILES in', os.path.dirname(h), '->', sorted(os.listdir(os.path.dirname(h))))
assert complete, "No COMPLETE causal_rag package found in the attached dataset: re-upload causal_rag_kaggle.zip as a new dataset version."
assert v11, "The mounted package is an OLD version (prompts.py has no prompt_graded; need code 1.3.0). Upload the latest causal_rag_kaggle.zip as a NEW VERSION of the dataset, STOP the session (power icon), re-attach the latest version, and start again."
pkg = v11[0]
for m in [m for m in sys.modules if m.startswith('causal_rag')]: del sys.modules[m]   # drop any stale import
sys.path.insert(0, os.path.dirname(os.path.dirname(pkg)))
print('USING:', pkg)
from causal_rag import paths
paths.restore_artifacts()          # pulls artifacts from any attached output-dataset of an earlier stage
SMOKE   = False                    # True = tiny REAL-data dry run (minutes) to verify the pipeline
N_EVAL  = 12 if SMOKE else 600     # applicants scored by the LLM systems (v1.0/v1.1 used 300; ~5 h at 600, ~6.5 h at 800)
CHUNK   = 6  if SMOKE else 100     # LLM stage saves every CHUNK applicants and resumes after a crash
RUN_NAME = 'v1_2_run'               # name of this run's folder in the exported results
AUDIT_SYSTEM = 'causal_rag_smooth'  # system used by the audit agent: 'causal_rag' (v1.1 prompt) or 'causal_rag_smooth'
EPOCHS  = 3  if SMOKE else 40
N_REPORT= 3  if SMOKE else 10      # applicants that get full DAG + HTML + rationale
WITH_ECONML = False                # EconML causal forest is very slow (hours on CPU); turn on only if you have the time
import torch
HAS_GPU = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if HAS_GPU else 'NONE - CPU ONLY')
REQUIRE_GPU = True                 # stages 03-08 need a GPU: on CPU, embeddings alone took ~6 h and CALF ~2.5 h
assert HAS_GPU or not REQUIRE_GPU or SMOKE, ("No GPU in this session. Settings -> Accelerator -> GPU T4 x2 "
                                             "(or your GPU quota is used up). Refusing to start a multi-hour CPU run.")
print('artifacts dir:', paths.ART, '| SMOKE =', SMOKE)

# CLIFF TEST v1.4 - a proper local test of "does the LLM react to the cutoff itself?"
The v1.3 "boundary flip" moved a field to just across its cutoff FROM WHEREVER IT WAS, so for applicants far from the cutoff it was a big move, not a nudge: absolute cliff ratios were confounded by move size (relative comparisons across systems stay valid).
This test fixes it. For each clause and applicant (all other fields unchanged) it asks the LLM at four values:
**thr-d, thr+d** (just across the cutoff) and **thr-5d, thr-3d** (an equally wide interval far from the cutoff, the placebo).
`cliff_index = mean|change across the cutoff| / mean|change over the placebo interval|`: about 1 = smooth reasoning, much greater than 1 = reacts to the cutoff / label.
Uses the saved evidence bundles; resumable; about 2 h for 150 applicants x 4 systems (estimate).

In [ ]:
import pickle, numpy as np, pandas as pd
need = ['evidence.pkl']
missing = [n for n in need if not paths.have(n)]
assert not missing, f"Missing {missing}: attach the artifacts dataset / notebook output of your run."
from causal_rag import pipeline as pl, eval as ev
from causal_rag.llm import LLM
N_CLIFF = 8 if SMOKE else 150
bundles = pickle.load(open(paths.art('evidence.pkl'), 'rb'))[:N_CLIFF]
llm = LLM()
SYS = ['no_rag', 'causal_conditions_only', 'causal_rag_smooth', 'causal_graded']
w = pl.run_cliff_stage(llm, bundles, SYS, chunk=25 if not SMOKE else 4)
w.to_parquet(paths.art('cliff_pairs.parquet'))
tab = ev.cliff_table(w, B=300 if SMOKE else 1000); display(tab)
bc = ev.cliff_table(w, by_clause=True, B=200 if SMOKE else 500); display(bc.pivot(index='clause', columns='system', values='cliff_index_abs'))
print('cliff_index ~1: smooth | >>1: reacts to the cutoff/label. 95% intervals are bootstrap over applicants.')

In [ ]:
import os
out = paths.ART / 'results_export' / 'v1_4_cliff_run'; out.mkdir(parents=True, exist_ok=True)
tab.to_csv(out / 'cliff_table.csv', index=False); bc.to_csv(out / 'cliff_table_by_clause.csv', index=False)
(out / 'RUN_INFO.md').write_text('# v1.4 local cliff test' + chr(10) + f'N={N_CLIFF} applicants, systems={SYS}, code 1.4.0' + chr(10))
print('exported to', out, '-> copy into the dataset under results/ to keep it')

### Optional: package this session's artifacts so they are never lost again
Download `crag_artifacts.zip` from the Output panel and upload it as a Kaggle Dataset (e.g. `crag-artifacts`). Attach that dataset in later notebooks: `restore_artifacts()` and the skip-if-exists guards pick everything up.

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/crag_artifacts', 'zip', '/kaggle/working', 'artifacts')
print('wrote /kaggle/working/crag_artifacts.zip')